# Stream depths — compute at 20 m, resample to coarser grids

Strategy:
1. Resample 0.5 m stream elevation to **20 m** only (using `min`).
2. Calculate stream depth at 20 m (`stream_ele - dem_clip`, clamped to `[-15, 0]`).
3. Resample the 20 m depth raster to 40 / 80 / 160 m using **mean** — so the depth values are consistent regardless of resolution.

In [1]:
import rasterio
import numpy as np
import os
import tools
from dotenv import load_dotenv

load_dotenv()
krycklan_data_path = os.getenv('KRYCKLAN_DATA_PATH')
krycklan_gis_data_path = os.getenv('KRYCKLAN_GIS_DATA_PATH')
print(f'Krycklan data path: {krycklan_data_path}')
print(f'Krycklan GIS data path: {krycklan_gis_data_path}')

file_ends = ['channels', '5haStreams']
paths = [
    os.path.join(krycklan_data_path, 'GIS', 'CATCHMENTS', 'ManualStreams', 'PROCESSED', '5haStreams_ManuallyDigitizedDitchesKrycklan_merged_5_0.5m_elev.tif'),
    os.path.join(krycklan_data_path, 'GIS', 'CATCHMENTS', 'Krycklan_5ha_streams', '5haStreams_0.5m_elev.tif'),
]

coarse_resos = [40, 80, 160]

catchs = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13,
                   141, 142, 143,
                   151, 152, 153,
                   161, 162, 163, 164,
                   20, 21, 22])

nodata_val = -9999

Krycklan data path: /Users/jpnousu/Library/CloudStorage/OneDrive-Luonnonvarakeskus/Krycklan data
Krycklan GIS data path: /Users/jpnousu/Data/Krycklan_GIS_data


In [2]:
# -----------------------------
# Step 1: Resample 0.5 m elevations to 20 m and mask with coarse raster
# (identical to stream_depths.ipynb but only for 20 m)
# -----------------------------
for elev_05m_fp, file in zip(paths, file_ends):
    for catch in catchs:
        outdir = os.path.join(krycklan_gis_data_path, '20m', f'C{catch}')
        os.makedirs(outdir, exist_ok=True)

        ref_raster_fp = os.path.join(outdir, f'{file}.asc')
        reproj_fp = os.path.join(outdir, f'{file}_elev.asc')

        # Resample 0.5 m raster to 20 m
        tools.reproj_match(
            infile=elev_05m_fp,
            match=ref_raster_fp,
            outfile=reproj_fp,
            resampling_method='min',
            save_in='asc'
        )

        # Apply coarse mask
        with rasterio.open(reproj_fp) as elev_src:
            elev_data = elev_src.read(1)
            elev_meta = elev_src.meta.copy()
            elev_nodata = elev_meta.get('nodata', nodata_val)

        with rasterio.open(ref_raster_fp) as mask_src:
            mask_data = mask_src.read(1)
            mask_meta = mask_src.meta.copy()
            mask_nodata = mask_meta.get('nodata', 0)

        mask_data[elev_data == elev_nodata] = mask_nodata
        with rasterio.open(ref_raster_fp, 'w', **mask_meta) as dst:
            dst.write(mask_data, 1)

        elev_data[mask_data == mask_nodata] = nodata_val
        elev_meta.update(nodata=nodata_val)
        with rasterio.open(reproj_fp, 'w', **elev_meta) as dst:
            dst.write(elev_data.astype(np.float32), 1)

Coregistered to shape: 69 41 
 Affine | 20.00, 0.00, 732306.00|
| 0.00,-20.00, 7134870.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 39 30 
 Affine | 20.00, 0.00, 730786.00|
| 0.00,-20.00, 7134570.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 13 21 
 Affine | 20.00, 0.00, 731326.00|
| 0.00,-20.00, 7134490.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 32 25 
 Affine | 20.00, 0.00, 730766.00|
| 0.00,-20.00, 7135130.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 77 63 
 Affine | 20.00, 0.00, 729606.00|
| 0.00,-20.00, 7136110.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 55 48 
 Affine | 20.00, 0.00, 730386.00|
| 0.00,-20.00, 7134710.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 52 24 
 Affine | 20.00, 0.00, 731086.00|
| 0.00,-20.00, 7134690.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 152 118 
 Affine | 20.00, 0.00, 728886.00|
| 0.00,-20.00, 7136250.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 108 72 
 Affine | 20.00, 0.00, 731006.00|
| 0.00,-20.00, 7134370.00|
| 0.00, 0.00, 1.00

In [3]:
# -----------------------------
# Step 2: Calculate stream depths at 20 m
# -----------------------------
for raster in file_ends:
    for catch in catchs:
        catch_dir = os.path.join(krycklan_gis_data_path, '20m', f'C{catch}')
        stream_ele_path = os.path.join(catch_dir, f'{raster}_elev.asc')
        dem_path = os.path.join(catch_dir, 'dem_clip.asc')
        stream_depth_path = os.path.join(catch_dir, f'{raster}_depth.asc')

        with rasterio.open(stream_ele_path) as stream_src:
            stream_ele = stream_src.read(1)
            stream_meta = stream_src.meta.copy()

        with rasterio.open(dem_path) as dem_src:
            dem_data = dem_src.read(1)

        dem_mask = (np.isfinite(dem_data)) & (dem_data != -9999)
        stream_mask = (stream_ele != -9999)

        stream_depth = np.where(dem_mask & stream_mask, stream_ele - dem_data, -9999)

        valid = stream_depth != -9999
        stream_depth[valid & (stream_depth > 0)] = 0
        stream_depth[valid & (stream_depth < -15)] = -15

        stream_meta.update(dtype=rasterio.float32, count=1, nodata=-9999)
        with rasterio.open(stream_depth_path, 'w', **stream_meta) as dst:
            dst.write(stream_depth.astype(rasterio.float32), 1)

In [4]:
# -----------------------------
# Step 3: Resample 20 m depth rasters to coarser resolutions using mean
# -----------------------------
for raster in file_ends:
    for catch in catchs:
        src_depth_path = os.path.join(krycklan_gis_data_path, '20m', f'C{catch}', f'{raster}_depth.asc')

        for reso in coarse_resos:
            coarse_dir = os.path.join(krycklan_gis_data_path, f'{reso}m', f'C{catch}')
            # Reference raster at target resolution for grid alignment
            ref_raster_fp = os.path.join(coarse_dir, f'{raster}.asc')
            out_depth_path = os.path.join(coarse_dir, f'{raster}_depth.asc')

            tools.reproj_match(
                infile=src_depth_path,
                match=ref_raster_fp,
                outfile=out_depth_path,
                resampling_method='average',
                save_in='asc'
            )

Coregistered to shape: 35 21 
 Affine | 40.00, 0.00, 732286.00|
| 0.00,-40.00, 7134870.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 18 11 
 Affine | 80.00, 0.00, 732246.00|
| 0.00,-80.00, 7134870.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 9 6 
 Affine | 160.00, 0.00, 732166.00|
| 0.00,-160.00, 7134870.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 20 16 
 Affine | 40.00, 0.00, 730766.00|
| 0.00,-40.00, 7134590.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 11 9 
 Affine | 80.00, 0.00, 730726.00|
| 0.00,-80.00, 7134630.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 6 5 
 Affine | 160.00, 0.00, 730726.00|
| 0.00,-160.00, 7134710.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 7 11 
 Affine | 40.00, 0.00, 731326.00|
| 0.00,-40.00, 7134510.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 4 6 
 Affine | 80.00, 0.00, 731286.00|
| 0.00,-80.00, 7134550.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 2 4 
 Affine | 160.00, 0.00, 731206.00|
| 0.00,-160.00, 7134550.00|
| 0.00, 0.00, 1.00|
Coreg